In [ ]:
!pip install kaggle
from google.colab import files
files.upload()  # (API key de tu cuenta Kaggle)
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kneroma/tacotrashdataset
!unzip -q tacotrashdataset.zip -d taco_data


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kneroma/tacotrashdataset
License(s): other
100% 2.79G/2.79G [02:30<00:00, 19.8MB/s]



In [ ]:
import json
with open('taco_data/data/annotations.json') as f:
    data = json.load(f)
from collections import Counter
cats = {c['id']: c['name'] for c in data['categories']}
counts = Counter(cats[ann['category_id']] for ann in data['annotations'])
print(counts.most_common(15))

[('Cigarette', 667), ('Unlabeled litter', 517), ('Plastic film', 451), ('Clear plastic bottle', 285), ('Other plastic', 273), ('Other plastic wrapper', 260), ('Drink can', 229), ('Plastic bottle cap', 209), ('Plastic straw', 157), ('Broken glass', 138), ('Styrofoam piece', 112), ('Glass bottle', 104), ('Disposable plastic cup', 104), ('Pop tab', 99), ('Other carton', 93)]


In [ ]:
import os, shutil

clases = {
    'Cigarette': 'colilla',
    'Plastic film': 'plastico_film',
    'Clear plastic bottle': 'botella_plastico'
}

for nombre_original, carpeta in clases.items():
    os.makedirs(f'dataset/train/{carpeta}', exist_ok=True)

img_by_id = {img['id']: img['file_name'] for img in data['images']}

contador = {c: 0 for c in clases}
for ann in data['annotations']:
    cat_name = cats[ann['category_id']]
    if cat_name in clases:
        carpeta = clases[cat_name]
        src = f"taco_data/data/{img_by_id[ann['image_id']]}"
        if os.path.exists(src):
            shutil.copy(src, f'dataset/train/{carpeta}/')
            contador[cat_name] += 1

print(contador)

{'Cigarette': 667, 'Plastic film': 451, 'Clear plastic bottle': 285}


In [ ]:
import requests

r = requests.get('https://api.inaturalist.org/v1/places/autocomplete', params={'q': 'Baja California'})
for p in r.json()['results']:
    print(p['id'], p['display_name'])

8501 Baja California, MX
7403 Baja California Sur, MX
207912 Río Vetas - Quebrada La Baja, Vereda Centro, California , ST, CO
187427 Honorable Ayuntamiento de La Paz, Baja California Sur
201982 Borderlands_Jacume Baja California
140842 Oasis San Ignacio, Municipio de Mulegé, Baja California Sur, Mex., MX
122739 Los Brasileros, Baja California Sur, Mexico
200674 Municipio de San Felipe, Baja California
208060 Área Influencia Rosario, Baja California Sur
75467 Islas del Alto Golfo de Baja California


In [ ]:
import os, requests

especies = {
    'Strongylocentrotus purpuratus': 'erizo_morado',   # nombre científico del erizo de mar morado
    'Zalophus californianus': 'lobo_marino'             # lobo marino de California
}

place_id =  8501 # ID de baja california

for nombre_cientifico, carpeta in especies.items():
    os.makedirs(f'dataset/train/{carpeta}', exist_ok=True)
    r = requests.get('https://api.inaturalist.org/v1/observations', params={
        'taxon_name': nombre_cientifico,
        'place_id': place_id,
        'quality_grade': 'research',
        'photos': 'true',
        'per_page': 50
    })
    resultados = r.json()['results']
    contador = 0
    for obs in resultados:
        for foto in obs.get('photos', []):
            url = foto['url'].replace('square', 'medium')  # pide resolución más grande
            try:
                img_data = requests.get(url, timeout=10).content
                with open(f'dataset/train/{carpeta}/{contador}.jpg', 'wb') as f:
                    f.write(img_data)
                contador += 1
            except Exception as e:
                print('Error con una foto:', e)
    print(f'{carpeta}: {contador} fotos descargadas')

erizo_morado: 68 fotos descargadas
lobo_marino: 123 fotos descargadas


In [ ]:
import random
for clase in os.listdir('dataset/train'):
    files_list = os.listdir(f'dataset/train/{clase}')
    val_files = random.sample(files_list, max(1, len(files_list)//5))
    os.makedirs(f'dataset/val/{clase}', exist_ok=True)
    for f in val_files:
        shutil.move(f'dataset/train/{clase}/{f}', f'dataset/val/{clase}/{f}')

In [ ]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/train',
    image_size=(224, 224),
    batch_size=8
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/val',
    image_size=(224, 224),
    batch_size=8
)

class_names = train_ds.class_names
print(class_names)  # confirmacion de  5 clases

Found 553 files belonging to 5 classes.
Found 135 files belonging to 5 classes.
['botella_plastico', 'colilla', 'erizo_morado', 'lobo_marino', 'plastico_film']


In [ ]:
from tensorflow.keras import layers, models

normalization = layers.Rescaling(1./127.5, offset=-1)
train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,293 (8.93 MB)

 Trainable params: 82,309 (321.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
#entrenando ahora si
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12
)

Epoch 1/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 63s 639ms/step - accuracy: 0.3472 - loss: 1.4339 - val_accuracy: 0.5333 - val_loss: 1.1722
Epoch 2/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 41s 254ms/step - accuracy: 0.5642 - loss: 1.0232 - val_accuracy: 0.5704 - val_loss: 1.0217
Epoch 3/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 283ms/step - accuracy: 0.6456 - loss: 0.8456 - val_accuracy: 0.5704 - val_loss: 0.9527
Epoch 4/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 18s 250ms/step - accuracy: 0.7071 - loss: 0.7433 - val_accuracy: 0.5778 - val_loss: 0.9091
Epoch 5/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 19s 275ms/step - accuracy: 0.7342 - loss: 0.6637 - val_accuracy: 0.5852 - val_loss: 0.8969
Epoch 6/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 18s 248ms/step - accuracy: 0.7559 - loss: 0.6021 - val_accuracy: 0.5778 - val_loss: 0.8847
Epoch 7/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 22s 275ms/step - accuracy: 0.7722 - loss: 0.5588 - val_accuracy: 0.5778 - val_loss: 0.8832
Epoch 8/12
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 277ms/step - accuracy: 0.7884 - loss: 0.5219 - val_accu

In [ ]:
model.save('modelo_prueba.h5')

In [ ]:
import numpy as np

for images, labels in val_ds.take(1):
    preds = model.predict(images)
    for i in range(3):
        print(class_names[np.argmax(preds[i])], '- confianza:', round(float(np.max(preds[i])), 3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
colilla - confianza: 0.499
lobo_marino - confianza: 0.95
lobo_marino - confianza: 0.994


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('modelo_prueba.tflite', 'wb') as f:
    f.write(tflite_model)

from google.colab import files
files.download('modelo_prueba.tflite')

Saved artifact at '/tmp/tmph4x6mn4w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  139431366546384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366547728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366544848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366546960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366545232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366546192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366543888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366545424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366547920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139431366548304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1394313665

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>